# Prediction Slice Explorer

Use this notebook to inspect a selected time window before trusting any low/high-frequency decomposition. Edit the parameter cell, run the command cell, then preview the generated curves.

The plot overlays ground truth, the current low-frequency moving average, and selected model predictions from BasicTS `test_results`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import json
import shlex
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "BasicTS").exists() and (path / "mvp_experiments").exists():
            return path
    for path in [Path("/home/yuzhang_fei/code/SpatialTemporalGraph"), Path("/Users/richardo/Desktop/STproject/SpatialTemporalGraph")]:
        if (path / "BasicTS").exists() and (path / "mvp_experiments").exists():
            return path
    raise FileNotFoundError("Cannot locate SpatialTemporalGraph repo root")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
MVP_ROOT = REPO_ROOT / "mvp_experiments/active/decoupled_spatiotemporal_diagnostics"
SCRIPT = MVP_ROOT / "scripts/plot_prediction_slice.py"
print(REPO_ROOT)
print(SCRIPT)

## Parameters

`start_mode` controls how the slice begins:

- `sample`: use `start_sample` on the saved `test_results` sample axis.
- `target_index`: use a global dataset target index.
- `datetime`: use absolute time; requires `dataset_start_datetime`.

In [ ]:
dataset_name = "SD"
search_root = REPO_ROOT / "BasicTS/checkpoints"
include = ["diagnostic_export_models_20260515"]
exclude = ["PropMLP"]

# Empty list means keep all discovered models. Otherwise use names such as:
# STID, DCRNN, STAEformer, GraphWaveNet, FEDformer, Crossformer, STGCN, Autoformer, ITransformer4D, PatchTST
models = ["STID", "DCRNN", "STAEformer", "GraphWaveNet"]

nodes = [0]
horizons = [12]
steps = 96

start_mode = "sample"  # sample | target_index | datetime
start_sample = 1200
start_target_index = 29256
dataset_start_datetime = None  # e.g. "2024-01-01 00:00"
start_datetime = None          # e.g. "2024-10-31 08:00"

low_window = 12
low_method = "centered"  # centered | trailing
x_axis = "target-index"  # sample | target-index | datetime
plot_format = "png"

output_dir = MVP_ROOT / "outputs/notebook_prediction_slices"
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--dataset-name", dataset_name,
    "--search-root", str(search_root),
    "--output-dir", str(output_dir),
    "--steps", str(steps),
    "--low-window", str(low_window),
    "--low-method", low_method,
    "--x-axis", x_axis,
    "--plot-format", plot_format,
]

for token in include:
    cmd.extend(["--include", token])
for token in exclude:
    cmd.extend(["--exclude", token])
if models:
    cmd.append("--models")
    cmd.extend(models)
cmd.append("--node")
cmd.extend(str(node) for node in nodes)
cmd.append("--horizon")
cmd.extend(str(horizon) for horizon in horizons)

if start_mode == "sample":
    cmd.extend(["--start-sample", str(start_sample)])
elif start_mode == "target_index":
    cmd.extend(["--start-target-index", str(start_target_index)])
elif start_mode == "datetime":
    if not dataset_start_datetime or not start_datetime:
        raise ValueError("datetime mode requires dataset_start_datetime and start_datetime")
    cmd.extend(["--dataset-start-datetime", dataset_start_datetime, "--start-datetime", start_datetime])
else:
    raise ValueError(f"Unknown start_mode: {start_mode}")

print(shlex.join(cmd))

In [ ]:
result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
pngs = sorted(output_dir.glob("prediction_slice_*.png"), key=lambda path: path.stat().st_mtime, reverse=True)
csvs = sorted(output_dir.glob("prediction_slice_*.csv"), key=lambda path: path.stat().st_mtime, reverse=True)
print(f"PNG files: {len(pngs)}")
for path in pngs[:12]:
    print(path)
    display(Image(filename=str(path)))

print(f"CSV files: {len(csvs)}")
for path in csvs[:12]:
    print(path)

In [ ]:
manifest_path = output_dir / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps({k: manifest[k] for k in ["dataset_name", "nodes", "horizons", "steps", "low_window", "low_method", "test_start_index", "input_len", "output_len", "frequency_minutes"]}, indent=2))